# Task 1 — Universal Multi-Corruption Denoising Autoencoder (v2: bottlenecked skips)

**v2 change.** In v1, Optuna selected full-width skip connections. A latent-ablation test showed that the decoder then ignored the latent: zeroing it cost only 0.4 dB, because the skips passed about 296k values, 6× the image size.
v2 squeezes every skip through a 1×1 convolution to 2–8 channels. It also enforces a **total code size** (latent + skips) of at most 1/8 of the input (6,144 values).
The final model is checked for its dependence on the latent, and the v1 wide-skip design is kept in the ablation as evidence.

One autoencoder restores clean, salt-and-pepper, blurred and occluded 128×128 pet images **without being told the corruption type**.

$\hat{x} = D(E(\tilde{x}))$, trained with $\mathcal{L} = \alpha\,\mathcal{L}_{1}(x,\hat{x}) + (1-\alpha)\,(1-\mathrm{SSIM}(x,\hat{x}))$, with $\alpha$ chosen by Optuna.

## ⚠️ Run in this order
1. **`MODE = "smoke"`** (the default) → *Run All* in the interactive editor. It needs about 5–10 min on a GPU.
   Every stage runs on tiny subsets: unit tests, an overfit-one-batch test, a 3-trial Optuna study, 3-epoch final training, test evaluation, all figures, ONNX export and parity, the skip ablation, and W&B logging.
   The notebook ends with `SMOKE TEST PASSED`. Check the figures and the W&B project (group `task1-v2-smoke`).
2. Change to **`MODE = "full"`** → **Save Version → Save & Run All (commit)**. This runs in the background, so you can close the browser.

**Kaggle settings (all checked by the pre-flight gate in Section 1, so nothing trains if one is missing):**
GPU **T4** · Internet **On** · *Add-ons → Secrets →* `WANDB_API_KEY` **ticked for this notebook** · Inputs: the Oxford-IIIT Pet dataset **and** the output of notebook 00.

| Section | Content |
|---|---|
| 0 | Setup, modules, config, W&B |
| 1 | Data (reuses notebook-00 split + manifests, or regenerates them identically) + **pre-flight gate** |
| 2 | Unit tests + overfit-one-batch test |
| 3 | Optuna study (SQLite storage, W&B run per trial, pruning) |
| 4 | Final training with best configuration (early stopping, checkpoints, W&B) |
| 5 | Test evaluation by corruption × severity |
| 6 | 12 representative examples + 4 failure cases (with error maps) |
| 7 | ONNX export, parity and latency |
| 8 | Skip-connection ablation incl. the original wide-skip design (code-dependence tests) |
| 9 | Summary |

## 0. Setup

In [ ]:
import importlib.util, subprocess, sys
missing = [p for p in ("onnx", "onnxruntime", "optuna") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
print("installed:", missing or "nothing needed")

In [ ]:
%%writefile pet_data.py
"""
pet_data.py — shared data pipeline for Tasks 1–3 (Oxford-IIIT Pet restoration).

Single source of truth for:
  * locating the Oxford-IIIT Pet dataset (Kaggle / torchvision layouts),
  * loading + standardising clean images (RGB, 128x128) with an on-disk cache
    of the *clean* resized images only (no corrupted copies are ever written),
  * the deterministic 80/20 train/val split of the official `trainval` list (seed 42),
  * the four input conditions and their corruption functions (pure NumPy, so the
    FastAPI backend can import them without PyTorch),
  * validation / test corruption manifests (deterministic, fully parameterised),
  * PyTorch datasets, a class-balanced batch sampler and DataLoader builders.

Label convention (used everywhere, incl. the Task 3 gate weight order w0..w3):
    0 = clean, 1 = salt_pepper, 2 = gaussian_blur, 3 = occlusion
"""
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
from PIL import Image

# --------------------------------------------------------------------------------------
# Constants (the assignment's corruption specification)
# --------------------------------------------------------------------------------------
IMG_SIZE = 128
SPLIT_SEED = 42
VAL_FRACTION = 0.20

CLEAN, SALT_PEPPER, BLUR, OCCLUSION = 0, 1, 2, 3
CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")
NUM_CLASSES = 4

SP_PROB_RANGE = (0.02, 0.15)          # uniform
BLUR_KERNELS = (3, 5, 7)              # uniform choice
BLUR_SIGMA_RANGE = (0.5, 2.5)         # uniform
OCC_BOX_RANGE = (1, 3)                # inclusive, uniform
OCC_COVERAGE_RANGE = (0.10, 0.35)     # union of boxes / image area, uniform
OCC_ASPECT_RANGE = (0.5, 2.0)         # box width/height, log-uniform (design choice)
OCC_MIN_SIDE = 4                      # px (design choice)
TEST_OCC_TOLERANCE = 0.01             # |actual - target| coverage for fixed test boxes

SEVERITY_LEVELS = ("low", "medium", "high")
TEST_SEVERITIES = {
    SALT_PEPPER: {"low": {"p": 0.03}, "medium": {"p": 0.08}, "high": {"p": 0.15}},
    BLUR: {"low": {"kernel_size": 3, "sigma": 0.7},
           "medium": {"kernel_size": 5, "sigma": 1.5},
           "high": {"kernel_size": 7, "sigma": 2.5}},
    OCCLUSION: {"low": {"n_boxes": 1, "coverage": 0.10},
                "medium": {"n_boxes": 2, "coverage": 0.20},
                "high": {"n_boxes": 3, "coverage": 0.35}},
}

MANIFEST_VERSION = 1


def corruption_config() -> dict:
    """The full corruption configuration as a JSON-serialisable dict (for W&B / report)."""
    return {
        "image_size": IMG_SIZE,
        "class_names": list(CLASS_NAMES),
        "train_class_probs": [0.25] * 4,
        "salt_pepper": {"p_uniform": list(SP_PROB_RANGE), "salt_vs_pepper": 0.5,
                        "granularity": "per-pixel (all 3 channels)"},
        "gaussian_blur": {"kernel_sizes": list(BLUR_KERNELS), "sigma_uniform": list(BLUR_SIGMA_RANGE),
                          "padding": "reflect", "separable": True},
        "occlusion": {"n_boxes": list(OCC_BOX_RANGE), "coverage_uniform": list(OCC_COVERAGE_RANGE),
                      "aspect_log_uniform": list(OCC_ASPECT_RANGE), "min_side_px": OCC_MIN_SIDE,
                      "boxes_overlap": False, "fill": 0.0},
        "test_severities": {CLASS_NAMES[k]: v for k, v in TEST_SEVERITIES.items()},
        "test_occlusion_tolerance": TEST_OCC_TOLERANCE,
    }


# --------------------------------------------------------------------------------------
# Dataset discovery and loading
# --------------------------------------------------------------------------------------
def read_split_list(path: str | Path) -> list[str]:
    """Read an official list file (e.g. 'Abyssinian_100 1 1 1'). Breed/species ids are discarded."""
    names = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            names.append(line.split()[0])
    return names


def find_pet_dataset(search_root: str | Path = "/kaggle/input") -> dict:
    """Locate annotations (trainval.txt + test.txt) and the images folder under `search_root`.

    Works for the common Kaggle upload (images/images, annotations/annotations) and the
    torchvision layout (oxford-iiit-pet/images, oxford-iiit-pet/annotations).
    """
    search_root = Path(search_root)
    ann_dir = None
    for tv in sorted(search_root.rglob("trainval.txt")):
        if (tv.parent / "test.txt").exists():
            ann_dir = tv.parent
            break
    if ann_dir is None:
        raise FileNotFoundError(
            f"No folder containing both trainval.txt and test.txt under {search_root}. "
            "The official split lists are required; set IMAGES_DIR / ANNOT_DIR manually.")
    first = read_split_list(ann_dir / "trainval.txt")[0]
    hits = sorted(search_root.rglob(f"{first}.jpg"))
    if not hits:
        raise FileNotFoundError(f"Could not find image '{first}.jpg' under {search_root}.")
    return {"annotations_dir": ann_dir, "images_dir": hits[0].parent}


def load_standardised_image(path: str | Path, size: int = IMG_SIZE) -> tuple[np.ndarray, str]:
    """Open any image, force 3-channel RGB, resize to size x size (bicubic, antialiased).

    Returns (uint8 HxWx3 array, original PIL mode). The backend must use this same function.
    """
    with Image.open(path) as im:
        mode = im.mode
        im = im.convert("RGB").resize((size, size), Image.Resampling.BICUBIC)
        return np.asarray(im, dtype=np.uint8).copy(), mode


def load_clean_images(names: Sequence[str], images_dir: str | Path, cache_path: str | Path | None = None,
                      size: int = IMG_SIZE, num_threads: int = 8, verbose: bool = True) -> dict:
    """Load + standardise every image in `names`. Caches the CLEAN resized uint8 array to .npz.

    Returns dict(images=(N,H,W,3) uint8, names=[...kept...], failed=[...], modes=Counter).
    Images that cannot be decoded are reported and dropped (the split is made afterwards).
    """
    names = list(names)
    if cache_path is not None and Path(cache_path).exists():
        z = np.load(cache_path, allow_pickle=False)
        if list(z["requested"]) == names and int(z["size"]) == size:
            out = {"images": z["images"], "names": list(z["names"]), "failed": list(z["failed"]),
                   "modes": Counter(json.loads(str(z["modes"]))), "from_cache": True}
            if verbose:
                print(f"Loaded {len(out['names'])} cached images from {cache_path}")
            return out

    images_dir = Path(images_dir)

    def _load(name):
        try:
            arr, mode = load_standardised_image(images_dir / f"{name}.jpg", size)
            return name, arr, mode, None
        except Exception as e:  # corrupted / unreadable file
            return name, None, None, repr(e)

    with ThreadPoolExecutor(max_workers=num_threads) as ex:
        results = list(ex.map(_load, names))

    kept = [(n, a, m) for n, a, m, err in results if err is None]
    failed = [(n, err) for n, a, m, err in results if err is not None]
    imgs = np.stack([a for _, a, _ in kept]) if kept else np.zeros((0, size, size, 3), np.uint8)
    out = {"images": imgs, "names": [n for n, _, _ in kept], "failed": [n for n, _ in failed],
           "modes": Counter(m for _, _, m in kept), "from_cache": False}
    if verbose:
        print(f"Loaded {len(kept)}/{len(names)} images; failed: {len(failed)}")
        for n, err in failed:
            print(f"  ! {n}: {err}")
    if cache_path is not None:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, images=imgs, names=np.array(out["names"]), requested=np.array(names),
                 failed=np.array(out["failed"], dtype=str), size=size, modes=json.dumps(dict(out["modes"])))
    return out


# --------------------------------------------------------------------------------------
# Deterministic split
# --------------------------------------------------------------------------------------
def make_split(names: Sequence[str], val_fraction: float = VAL_FRACTION, seed: int = SPLIT_SEED) -> dict:
    """80/20 random split of the official trainval list with numpy default_rng(seed)."""
    names = list(names)
    n = len(names)
    perm = np.random.default_rng(seed).permutation(n)
    n_val = int(round(val_fraction * n))
    n_train = n - n_val
    train = [names[i] for i in sorted(perm[:n_train])]
    val = [names[i] for i in sorted(perm[n_train:])]
    return {"seed": seed, "val_fraction": val_fraction, "method": "numpy.default_rng(seed).permutation",
            "n_total": n, "train": train, "val": val}


# --------------------------------------------------------------------------------------
# Corruption functions (pure NumPy; img = float32 HxWx3 in [0,1])
# --------------------------------------------------------------------------------------
def salt_and_pepper(img: np.ndarray, p: float, seed: int) -> np.ndarray:
    """Each pixel is selected with prob p; selected pixels become white or black (50/50)."""
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    selected = rng.random((h, w)) < p
    salt = rng.random((h, w)) < 0.5
    out = img.copy()
    out[selected & salt] = 1.0
    out[selected & ~salt] = 0.0
    return out


def gaussian_kernel1d(kernel_size: int, sigma: float) -> np.ndarray:
    x = np.arange(kernel_size, dtype=np.float64) - (kernel_size - 1) / 2.0
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return (k / k.sum()).astype(np.float32)


def gaussian_blur(img: np.ndarray, kernel_size: int, sigma: float) -> np.ndarray:
    """Separable Gaussian blur with reflect padding (matches torchvision's gaussian_blur)."""
    k = gaussian_kernel1d(kernel_size, sigma)
    r = kernel_size // 2
    h, w = img.shape[:2]
    padded = np.pad(img, ((r, r), (r, r), (0, 0)), mode="reflect")
    horiz = sum(k[i] * padded[:, i:i + w] for i in range(kernel_size))        # (h+2r, w, 3)
    out = sum(k[i] * horiz[i:i + h] for i in range(kernel_size))              # (h, w, 3)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def occlude(img: np.ndarray, boxes: Iterable[Sequence[int]], fill: float = 0.0) -> np.ndarray:
    """Black out boxes given as [x0, y0, x1, y1] (x1/y1 exclusive)."""
    out = img.copy()
    for x0, y0, x1, y1 in boxes:
        out[y0:y1, x0:x1] = fill
    return out


def boxes_coverage(boxes, h: int = IMG_SIZE, w: int = IMG_SIZE) -> float:
    mask = np.zeros((h, w), bool)
    for x0, y0, x1, y1 in boxes:
        mask[y0:y1, x0:x1] = True
    return float(mask.mean())


def sample_occlusion_boxes(rng: np.random.Generator, n_boxes: int, coverage: float,
                           h: int = IMG_SIZE, w: int = IMG_SIZE,
                           accept=None, max_attempts: int = 1000) -> tuple[list[list[int]], float]:
    """Sample `n_boxes` non-overlapping boxes whose union covers ~`coverage` of the image.

    The total area is split between boxes (each gets at least half of an equal share), each
    box gets a log-uniform aspect ratio, and boxes are placed at random non-overlapping
    positions. Non-overlap makes the union area equal to the sum of box areas, so the jointly
    covered fraction is controlled precisely. `accept(actual)->bool` decides if rounding is OK.
    """
    if accept is None:
        accept = lambda a: abs(a - coverage) <= TEST_OCC_TOLERANCE  # noqa: E731
    total = coverage * h * w
    lo_ar, hi_ar = np.log(OCC_ASPECT_RANGE[0]), np.log(OCC_ASPECT_RANGE[1])
    for _ in range(max_attempts):
        shares = 0.5 / n_boxes + 0.5 * rng.dirichlet([2.0] * n_boxes)
        boxes, ok = [], True
        for area in shares * total:
            ar = float(np.exp(rng.uniform(lo_ar, hi_ar)))
            bw = int(np.clip(round(np.sqrt(area * ar)), OCC_MIN_SIDE, w))
            bh = int(np.clip(round(area / bw), OCC_MIN_SIDE, h))
            for _try in range(200):
                x0 = int(rng.integers(0, w - bw + 1))
                y0 = int(rng.integers(0, h - bh + 1))
                cand = [x0, y0, x0 + bw, y0 + bh]
                if all(cand[2] <= b[0] or b[2] <= cand[0] or cand[3] <= b[1] or b[3] <= cand[1]
                       for b in boxes):
                    boxes.append(cand)
                    break
            else:
                ok = False
                break
        if ok:
            actual = boxes_coverage(boxes, h, w)
            if accept(actual):
                return boxes, actual
    raise RuntimeError(f"Could not place {n_boxes} boxes covering {coverage:.3f}")


def _new_seed(rng: np.random.Generator) -> int:
    return int(rng.integers(0, 2**31 - 1))


def sample_params(label: int, rng: np.random.Generator) -> dict:
    """Sample corruption parameters from the TRAINING distribution. Fully reproducible via 'seed'."""
    seed = _new_seed(rng)
    r = np.random.default_rng(seed)
    label = int(label)
    if label == CLEAN:
        return {"label": CLEAN, "type": "clean", "seed": seed}
    if label == SALT_PEPPER:
        return {"label": SALT_PEPPER, "type": "salt_pepper", "seed": seed,
                "p": float(r.uniform(*SP_PROB_RANGE))}
    if label == BLUR:
        return {"label": BLUR, "type": "gaussian_blur", "seed": seed,
                "kernel_size": int(r.choice(BLUR_KERNELS)), "sigma": float(r.uniform(*BLUR_SIGMA_RANGE))}
    if label == OCCLUSION:
        n = int(r.integers(OCC_BOX_RANGE[0], OCC_BOX_RANGE[1] + 1))
        cov = float(r.uniform(*OCC_COVERAGE_RANGE))
        lo, hi = OCC_COVERAGE_RANGE
        boxes, actual = sample_occlusion_boxes(r, n, cov, accept=lambda a: lo <= a <= hi)
        return {"label": OCCLUSION, "type": "occlusion", "seed": seed, "n_boxes": n,
                "coverage_target": cov, "coverage": actual, "boxes": boxes}
    raise ValueError(f"Unknown label {label}")


def fixed_params(label: int, level: str, seed: int) -> dict:
    """Parameters for the fixed TEST severities (low/medium/high)."""
    label = int(label)
    if label == CLEAN:
        return {"label": CLEAN, "type": "clean", "seed": seed}
    spec = TEST_SEVERITIES[label][level]
    base = {"label": label, "type": CLASS_NAMES[label], "seed": seed}
    if label == SALT_PEPPER:
        return {**base, "p": spec["p"]}
    if label == BLUR:
        return {**base, "kernel_size": spec["kernel_size"], "sigma": spec["sigma"]}
    boxes, actual = sample_occlusion_boxes(np.random.default_rng(seed), spec["n_boxes"], spec["coverage"])
    return {**base, "n_boxes": spec["n_boxes"], "coverage_target": spec["coverage"],
            "coverage": actual, "boxes": boxes}


def apply_corruption(img: np.ndarray, params: dict) -> np.ndarray:
    """Deterministically apply a parameter dict (from sample_params / fixed_params / a manifest)."""
    label = int(params["label"])
    if label == CLEAN:
        return img.copy()
    if label == SALT_PEPPER:
        return salt_and_pepper(img, params["p"], params["seed"])
    if label == BLUR:
        return gaussian_blur(img, params["kernel_size"], params["sigma"])
    if label == OCCLUSION:
        return occlude(img, params["boxes"])
    raise ValueError(f"Unknown label {label}")


def describe_params(params: dict) -> str:
    """Short human-readable description (plot titles, API responses)."""
    t = int(params["label"])
    if t == CLEAN:
        return "clean"
    if t == SALT_PEPPER:
        return f"s&p p={params['p']:.3f}"
    if t == BLUR:
        return f"blur k={params['kernel_size']} σ={params['sigma']:.2f}"
    return f"occl {params['n_boxes']}box {100 * params['coverage']:.1f}%"


# --------------------------------------------------------------------------------------
# Manifests
# --------------------------------------------------------------------------------------
def build_val_manifest(val_names: Sequence[str], seed: int = SPLIT_SEED) -> dict:
    """Exactly balanced (25% per condition) validation manifest; params from the training ranges."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 1]))
    n = len(val_names)
    labels = rng.permutation(np.resize(np.arange(NUM_CLASSES), n))
    records = []
    for i, (name, label) in enumerate(zip(val_names, labels)):
        p = sample_params(int(label), rng)
        records.append({"id": f"{name}__{p['type']}", "image": name, "index": i,
                        "severity_level": "sampled" if label != CLEAN else "none", **p})
    return {"meta": {"kind": "validation", "version": MANIFEST_VERSION, "seed": seed,
                     "seed_sequence": [seed, 1], "n_images": n, "n_records": len(records),
                     "config": corruption_config()},
            "records": records}


def build_test_manifest(test_names: Sequence[str], seed: int = SPLIT_SEED) -> dict:
    """Every test image × {clean, 3 corruptions × 3 fixed severities} = 10 records per image."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 2]))
    records = []
    for i, name in enumerate(test_names):
        s = _new_seed(rng)
        records.append({"id": f"{name}__clean", "image": name, "index": i,
                        "severity_level": "none", **fixed_params(CLEAN, "none", s)})
        for label in (SALT_PEPPER, BLUR, OCCLUSION):
            for level in SEVERITY_LEVELS:
                s = _new_seed(rng)
                p = fixed_params(label, level, s)
                records.append({"id": f"{name}__{p['type']}__{level}", "image": name, "index": i,
                                "severity_level": level, **p})
    return {"meta": {"kind": "test", "version": MANIFEST_VERSION, "seed": seed,
                     "seed_sequence": [seed, 2], "n_images": len(test_names), "n_records": len(records),
                     "config": corruption_config()},
            "records": records}


def save_json(obj, path: str | Path, indent=None) -> str:
    """Write JSON deterministically and return its SHA-256 (use it to prove files are identical)."""
    text = json.dumps(obj, indent=indent, sort_keys=True, separators=None if indent else (",", ":"))
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(text)
    return hashlib.sha256(text.encode()).hexdigest()


def load_json(path: str | Path):
    return json.loads(Path(path).read_text())


def sha256_file(path: str | Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# --------------------------------------------------------------------------------------
# PyTorch datasets / samplers / loaders  (torch imported lazily so the backend stays light)
# --------------------------------------------------------------------------------------
try:
    import torch
    from torch.utils.data import DataLoader, Dataset, Sampler, get_worker_info
except ImportError:  # backend without torch: only the NumPy functions above are needed
    torch = None
    Dataset = object
    Sampler = object


def to_chw_tensor(img_hwc: np.ndarray):
    return torch.from_numpy(np.ascontiguousarray(img_hwc.transpose(2, 0, 1), dtype=np.float32))


class RuntimeCorruptionDataset(Dataset):
    """Training dataset: a NEW corruption type + severity is sampled on every __getitem__.

    Returns (corrupted_image, clean_image, corruption_label):
        (3,128,128) float32 [0,1], (3,128,128) float32 [0,1], int64 scalar.
    `classes` restricts which conditions are drawn (e.g. (1,) for the Task 2 salt-and-pepper
    specialist); they are drawn uniformly. `item` may also be an (index, label) tuple, which is
    how BalancedCorruptionBatchSampler forces exactly balanced batches.
    """

    def __init__(self, images_u8: np.ndarray, classes: Sequence[int] = (0, 1, 2, 3), seed: int | None = None):
        self.images = images_u8
        self.classes = np.asarray(classes, dtype=np.int64)
        self.seed = seed
        self._rng, self._rng_key = None, None

    def __len__(self):
        return len(self.images)

    def _get_rng(self) -> np.random.Generator:
        # One independent generator per process/worker. Worker seeds come from torch
        # (base_seed + worker_id, re-drawn every epoch), so torch.manual_seed controls everything
        # and forked workers never share a random stream.
        info = get_worker_info()
        key = (os.getpid(), info.seed if info is not None else None)
        if key != self._rng_key:
            if info is not None:
                s = info.seed % 2**32
            else:
                s = self.seed if self.seed is not None else int(torch.initial_seed() % 2**32)
            self._rng, self._rng_key = np.random.default_rng(s), key
        return self._rng

    def get_with_params(self, item):
        rng = self._get_rng()
        if isinstance(item, (tuple, list)):
            idx, label = int(item[0]), int(item[1])
        else:
            idx, label = int(item), int(rng.choice(self.classes))
        clean = self.images[idx].astype(np.float32) / 255.0
        params = sample_params(label, rng)
        corrupted = apply_corruption(clean, params)
        return corrupted, clean, params

    def __getitem__(self, item):
        corrupted, clean, params = self.get_with_params(item)
        return to_chw_tensor(corrupted), to_chw_tensor(clean), torch.tensor(params["label"], dtype=torch.long)


class ManifestCorruptionDataset(Dataset):
    """Validation/test dataset: corruption parameters are read from a fixed manifest.

    `name_to_row` maps image name -> row in `images_u8`. With return_index=True a 4th element
    (the record index) is returned so results can be grouped by type/severity afterwards.
    """

    def __init__(self, images_u8: np.ndarray, name_to_row: dict, records: list[dict],
                 classes: Sequence[int] | None = None, return_index: bool = False):
        self.images = images_u8
        self.name_to_row = name_to_row
        self.records = [r for r in records if classes is None or r["label"] in classes]
        self.return_index = return_index

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        clean = self.images[self.name_to_row[rec["image"]]].astype(np.float32) / 255.0
        corrupted = apply_corruption(clean, rec)
        out = (to_chw_tensor(corrupted), to_chw_tensor(clean), torch.tensor(rec["label"], dtype=torch.long))
        return out + (torch.tensor(i),) if self.return_index else out


class BalancedCorruptionBatchSampler(Sampler):
    """Yields batches of (image_index, label) with EXACTLY equal counts per class
    (when batch_size is a multiple of the number of classes). Intended for the Task 2 classifier.
    """

    def __init__(self, n_items: int, batch_size: int, classes: Sequence[int] = (0, 1, 2, 3),
                 drop_last: bool = True, seed: int | None = None):
        self.n, self.bs, self.classes, self.drop_last = n_items, batch_size, np.asarray(classes), drop_last
        self.seed, self.epoch = seed, 0

    def __len__(self):
        return self.n // self.bs if self.drop_last else -(-self.n // self.bs)

    def __iter__(self):
        base = self.seed if self.seed is not None else int(torch.randint(0, 2**31 - 1, ()).item())
        rng = np.random.default_rng([base, self.epoch])
        self.epoch += 1
        perm = rng.permutation(self.n)
        for start in range(0, len(self) * self.bs, self.bs):
            idx = perm[start:start + self.bs]
            labels = rng.permutation(np.resize(self.classes, len(idx)))
            yield [(int(i), int(l)) for i, l in zip(idx, labels)]


def default_num_workers() -> int:
    return max(0, min(4, (os.cpu_count() or 2)))


def make_train_loader(images_u8, batch_size=32, classes=(0, 1, 2, 3), balanced=False,
                      num_workers=None, seed=None, pin_memory=True, drop_last=True):
    num_workers = default_num_workers() if num_workers is None else num_workers
    ds = RuntimeCorruptionDataset(images_u8, classes=classes, seed=seed)
    kw = dict(num_workers=num_workers, pin_memory=pin_memory and torch.cuda.is_available(),
              persistent_workers=num_workers > 0)
    if balanced:
        bs = BalancedCorruptionBatchSampler(len(ds), batch_size, classes, drop_last=drop_last, seed=seed)
        return DataLoader(ds, batch_sampler=bs, **kw)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=drop_last, **kw)


def make_eval_loader(images_u8, name_to_row, records, batch_size=64, classes=None,
                     return_index=False, num_workers=None, pin_memory=True):
    num_workers = default_num_workers() if num_workers is None else num_workers
    ds = ManifestCorruptionDataset(images_u8, name_to_row, records, classes=classes, return_index=return_index)
    # Not persistent: eval workers exit after each pass, so idle loaders don't hold RAM
    # (each worker gradually copies the Python objects it touches — "copy-on-read").
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                      pin_memory=pin_memory and torch.cuda.is_available(), persistent_workers=False)


class PetData:
    """Everything a Task 1–3 notebook needs, built by prepare_data()."""

    def __init__(self, **kw):
        self.__dict__.update(kw)

    def load_test_images(self, cache_dir=None):
        """FINAL EVALUATION ONLY: decode the quarantined official test images (cached)."""
        cache = Path(cache_dir or self.cache_dir) / "test_128.npz"
        out = load_clean_images(self.test_names, self.images_dir, cache_path=cache)
        return out["images"], {n: i for i, n in enumerate(out["names"])}, out["failed"]

    def summary(self) -> dict:
        return {"source": self.source, "images_dir": str(self.images_dir), "n_train": len(self.train_images),
                "n_val": len(self.val_images), "n_test": len(self.test_names),
                "n_val_records": len(self.val_manifest["records"]),
                "n_test_records": len(self.test_manifest["records"]),
                "split_sha256": self.hashes.get("split"), "val_manifest_sha256": self.hashes.get("val"),
                "test_manifest_sha256": self.hashes.get("test")}


def find_prep_outputs(search_root="/kaggle/input") -> Path | None:
    """Locate an attached output of notebook 00 (folder containing manifests/val_manifest.json)."""
    for p in sorted(Path(search_root).rglob("val_manifest.json")):
        root = p.parent.parent
        if (root / "manifests" / "test_manifest.json").exists() and (root / "artifacts" / "split_seed42.json").exists():
            return root
    return None


def prepare_data(input_root="/kaggle/input", work_dir="/kaggle/working", images_dir=None, annot_dir=None,
                 seed: int = SPLIT_SEED, verbose: bool = True) -> PetData:
    """Return split, clean train/val arrays and both manifests.

    1. If notebook 00's output is attached under `input_root`, its split + manifests are reused
       (guarantees byte-identical data across Tasks 1–3).
    2. Otherwise everything is regenerated deterministically (same seeds -> same files/hashes)
       and saved under `work_dir`.
    The official Oxford-IIIT Pet dataset must be attached in both cases (for the pixels).
    """
    input_root, work_dir = Path(input_root), Path(work_dir)
    cache_dir = work_dir / "cache"
    if images_dir is None or annot_dir is None:
        found = find_pet_dataset(input_root)
        images_dir = Path(images_dir or found["images_dir"])
        annot_dir = Path(annot_dir or found["annotations_dir"])
    test_names = read_split_list(Path(annot_dir) / "test.txt")

    prep = find_prep_outputs(input_root)
    if prep is not None:
        source = f"attached notebook-00 output: {prep}"
        split = load_json(prep / "artifacts" / "split_seed42.json")
        val_manifest = load_json(prep / "manifests" / "val_manifest.json")
        test_manifest = load_json(prep / "manifests" / "test_manifest.json")
        hashes = {"split": sha256_file(prep / "artifacts" / "split_seed42.json"),
                  "val": sha256_file(prep / "manifests" / "val_manifest.json"),
                  "test": sha256_file(prep / "manifests" / "test_manifest.json")}
        wanted = split["train"] + split["val"]
        cached = prep / "cache" / "trainval_128.npz"
        tv = None
        if cached.exists():
            z = load_clean_images(read_split_list(Path(annot_dir) / "trainval.txt"), images_dir,
                                  cache_path=cached, verbose=False)
            tv = z if set(wanted) <= set(z["names"]) else None
        if tv is None:
            tv = load_clean_images(wanted, images_dir, cache_path=cache_dir / "trainval_128.npz", verbose=verbose)
    else:
        source = "regenerated deterministically (notebook-00 output not attached)"
        trainval_names = read_split_list(Path(annot_dir) / "trainval.txt")
        tv = load_clean_images(trainval_names, images_dir, cache_path=cache_dir / "trainval_128.npz", verbose=verbose)
        split = make_split(tv["names"], VAL_FRACTION, seed)
        split["excluded_unreadable"] = tv["failed"]
        val_manifest = build_val_manifest(split["val"], seed)
        test_manifest = build_test_manifest(test_names, seed)
        hashes = {"split": save_json(split, work_dir / "artifacts" / "split_seed42.json", indent=1),
                  "val": save_json(val_manifest, work_dir / "manifests" / "val_manifest.json"),
                  "test": save_json(test_manifest, work_dir / "manifests" / "test_manifest.json")}

    row = {n: i for i, n in enumerate(tv["names"])}
    train_images = tv["images"][[row[n] for n in split["train"]]]
    val_images = tv["images"][[row[n] for n in split["val"]]]
    assert not set(split["train"]) & set(split["val"]) and not (set(split["train"]) | set(split["val"])) & set(test_names)
    data = PetData(source=source, images_dir=images_dir, annot_dir=annot_dir, cache_dir=cache_dir, split=split,
                   train_images=train_images, val_images=val_images,
                   val_name_to_row={n: i for i, n in enumerate(split["val"])},
                   val_manifest=val_manifest, test_manifest=test_manifest, test_names=test_names, hashes=hashes)
    if verbose:
        for k, v in data.summary().items():
            print(f"  {k:22s} {v}")
    return data


def seed_everything(seed: int = SPLIT_SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile restoration.py
"""
restoration.py — shared restoration components (Task 1, reused by Task 2 specialists and Task 3).

Contents
  * SSIM (Gaussian window, Wang et al. 2004) + PSNR, per-image and batched, differentiable
  * L1 + SSIM loss:  L = alpha * L1 + (1 - alpha) * (1 - SSIM)
  * ConvAutoencoder: conv encoder -> compressed spatial latent (1x1-conv bottleneck + dropout)
                     -> conv decoder, with optional LIMITED skip connections (deepest levels only)
  * train_one_epoch / evaluate (AMP-aware, per-corruption metrics)
  * ONNX export, PyTorch-vs-ONNX parity check, latency measurement
  * plotting helpers for restoration grids (target / input / output / |error|)
"""
from __future__ import annotations

import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASS_NAMES = ("clean", "salt_pepper", "gaussian_blur", "occlusion")


# ======================================================================================
# Metrics
# ======================================================================================
def _gaussian_window(size: int, sigma: float, channels: int, device, dtype):
    x = torch.arange(size, device=device, dtype=dtype) - (size - 1) / 2
    g = torch.exp(-(x ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w2d = g[:, None] * g[None, :]
    return w2d.expand(channels, 1, size, size).contiguous()


def ssim_map(x: torch.Tensor, y: torch.Tensor, window_size: int = 11, sigma: float = 1.5,
             data_range: float = 1.0) -> torch.Tensor:
    """SSIM map (B,C,H',W') with an 11x11 Gaussian window (sigma 1.5), 'valid' region only.
    Matches skimage.structural_similarity(gaussian_weights=True, sigma=1.5,
    use_sample_covariance=False, data_range=1)."""
    x, y = x.float(), y.float()
    c = x.shape[1]
    w = _gaussian_window(window_size, sigma, c, x.device, x.dtype)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    mu_x = F.conv2d(x, w, groups=c)
    mu_y = F.conv2d(y, w, groups=c)
    sxx = F.conv2d(x * x, w, groups=c) - mu_x ** 2
    syy = F.conv2d(y * y, w, groups=c) - mu_y ** 2
    sxy = F.conv2d(x * y, w, groups=c) - mu_x * mu_y
    return ((2 * mu_x * mu_y + c1) * (2 * sxy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sxx + syy + c2))


def ssim_per_image(x, y, **kw) -> torch.Tensor:
    return ssim_map(x, y, **kw).mean(dim=(1, 2, 3))


def psnr_per_image(x, y, data_range: float = 1.0) -> torch.Tensor:
    mse = ((x.float() - y.float()) ** 2).mean(dim=(1, 2, 3)).clamp_min(1e-10)
    return 10 * torch.log10(data_range ** 2 / mse)


def mae_per_image(x, y) -> torch.Tensor:
    return (x.float() - y.float()).abs().mean(dim=(1, 2, 3))


def val_score(psnr_mean: float, ssim_mean: float) -> float:
    """Optuna objective (alpha-independent, so the study cannot 'win' by changing the loss scale):
    equal-weight combination of structural similarity and reconstruction fidelity (PSNR/40, capped)."""
    return 0.5 * float(ssim_mean) + 0.5 * min(float(psnr_mean) / 40.0, 1.0)


# ======================================================================================
# Loss
# ======================================================================================
class L1SSIMLoss(nn.Module):
    """alpha * L1 + (1 - alpha) * (1 - SSIM). Always computed in float32 (AMP-safe)."""

    def __init__(self, alpha: float = 0.8):
        super().__init__()
        self.alpha = float(alpha)

    def forward(self, pred, target):
        pred, target = pred.float(), target.float()
        l1 = F.l1_loss(pred, target)
        s = ssim_map(pred, target).mean()
        total = self.alpha * l1 + (1 - self.alpha) * (1 - s)
        return total, {"l1": l1.detach(), "ssim": s.detach()}


# ======================================================================================
# Model
# ======================================================================================
def norm(c: int) -> nn.GroupNorm:
    # GroupNorm instead of BatchNorm: no running statistics, so train- and eval-mode behave the same
    # at any batch size (Optuna samples batch sizes 16-64), and frozen experts in Task 3 cannot drift.
    return nn.GroupNorm(min(8, c), c)


def conv_block(cin: int, cout: int, stride: int = 1) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, stride, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
        nn.Conv2d(cout, cout, 3, 1, 1, bias=False), norm(cout), nn.SiLU(inplace=True),
    )


class ConvAutoencoder(nn.Module):
    """Convolutional denoising autoencoder with a genuine compressed code.

    Encoder: 128 -> 64 -> 32 -> 16 -> 8 (stride-2 conv blocks), channels grow base x (1,2,4,8,8).
    Latent:  1x1 conv to `latent_channels` at 8x8 (e.g. 16 ch -> 1,024 values), then code dropout.
    Decoder: bilinear upsample + conv blocks (no transposed-conv checkerboard artefacts), sigmoid output.

    Skip connections (skip_levels = 0, 1 or 2) are BOTTLENECKED: the encoder feature at the
    16x16 (and 32x32) level is squeezed by a 1x1 conv to `skip_channels` channels and passes the same
    code dropout before reaching the decoder. The information the decoder receives is therefore
    bounded by the total code size

        code_dim = latent_dim + sum_over_skipped_levels(skip_channels * H_l * W_l),

    which is what `compression_ratio` reports (input values / code values). Full- and half-resolution
    features are never skipped. skip_channels=None reproduces the original full-width skips
    (e.g. 384 x 16 x 16 = 98,304 values, larger than the image); it is kept only for the ablation,
    which showed that such skips let the decoder ignore the latent.
    """

    def __init__(self, base_channels: int = 32, latent_channels: int = 32, dropout: float = 0.1,
                 skip_levels: int = 0, skip_channels: int | None = 4, depth: int = 4, max_mult: int = 8,
                 img_size: int = 128):
        super().__init__()
        assert 0 <= skip_levels <= 2, "only the two deepest levels may be skipped"
        self.config = dict(base_channels=base_channels, latent_channels=latent_channels, dropout=dropout,
                           skip_levels=skip_levels, skip_channels=skip_channels, depth=depth,
                           max_mult=max_mult, img_size=img_size)
        chs = [min(base_channels * 2 ** i, base_channels * max_mult) for i in range(depth + 1)]
        self.chs, self.depth, self.skip_levels, self.img_size = chs, depth, skip_levels, img_size
        self.stem = conv_block(3, chs[0], 1)
        self.down = nn.ModuleList([conv_block(chs[i], chs[i + 1], 2) for i in range(depth)])
        self.to_latent = nn.Conv2d(chs[-1], latent_channels, 1)
        self.code_dropout = nn.Dropout(dropout)
        self.from_latent = nn.Sequential(nn.Conv2d(latent_channels, chs[-1], 1, bias=False),
                                         norm(chs[-1]), nn.SiLU(inplace=True))
        # decoder stage j upsamples to resolution level (depth - j - 1); skips only for j < skip_levels
        self.skip_proj = nn.ModuleList()
        self.skip_shapes = []
        self.up = nn.ModuleList()
        for j in range(depth):
            lvl = depth - j - 1
            extra = 0
            if j < skip_levels:
                width = chs[lvl] if skip_channels is None else skip_channels
                self.skip_proj.append(nn.Identity() if skip_channels is None else nn.Conv2d(chs[lvl], skip_channels, 1))
                res = img_size // 2 ** lvl
                self.skip_shapes.append((width, res, res))
                extra = width
            self.up.append(conv_block(chs[lvl + 1] + extra, chs[lvl], 1))
        self.head = nn.Conv2d(chs[0], 3, 3, 1, 1)
        self.latent_shape = (latent_channels, img_size // 2 ** depth, img_size // 2 ** depth)

    # ---- code-size accounting ------------------------------------------------------------
    @property
    def latent_dim(self) -> int:
        return int(np.prod(self.latent_shape))

    @property
    def skip_dim(self) -> int:
        return int(sum(np.prod(s) for s in self.skip_shapes))

    @property
    def code_dim(self) -> int:
        """Total number of values the decoder receives from the encoder (latent + skip codes)."""
        return self.latent_dim + self.skip_dim

    @property
    def compression_ratio(self) -> float:
        return 3 * self.img_size ** 2 / self.code_dim

    # ---- forward -------------------------------------------------------------------------
    def encode(self, x):
        """Returns (latent, skip_codes). skip_codes[j] feeds decoder stage j (deepest first)."""
        feats = []
        h = self.stem(x)
        for blk in self.down:
            feats.append(h)
            h = blk(h)
        skips = [proj(feats[self.depth - j - 1]) for j, proj in enumerate(self.skip_proj)]
        return self.to_latent(h), skips

    def decode(self, z, skips=None):
        h = self.from_latent(self.code_dropout(z))
        for j, blk in enumerate(self.up):
            h = F.interpolate(h, scale_factor=2, mode="bilinear", align_corners=False)
            if j < self.skip_levels:
                s = skips[j] if skips is not None else h.new_zeros(h.shape[0], *self.skip_shapes[j])
                h = torch.cat([h, self.code_dropout(s)], dim=1)
            h = blk(h)
        return torch.sigmoid(self.head(h))

    def forward(self, x):
        z, skips = self.encode(x)
        return self.decode(z, skips)


@torch.no_grad()
def code_dependence(model, loader, device, max_batches=None) -> dict:
    """How much the output depends on each part of the code (evaluated on a manifest loader).

    full:        normal forward pass
    zero_latent: latent replaced by zeros (skips intact)   -> large drop = the latent is used
    zero_skips:  skip codes replaced by zeros (latent intact) -> drop = what the skips contribute
    """
    model.eval()
    acc = {k: {"psnr": [], "ssim": []} for k in ("full", "zero_latent", "zero_skips")}
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device), batch[1].to(device)
        z, skips = model.encode(x)
        outs = {"full": model.decode(z, skips), "zero_latent": model.decode(torch.zeros_like(z), skips)}
        if model.skip_levels:
            outs["zero_skips"] = model.decode(z, [torch.zeros_like(s) for s in skips])
        for k, o in outs.items():
            o = o.float().clamp(0, 1)
            acc[k]["psnr"].append(psnr_per_image(o, y).cpu())
            acc[k]["ssim"].append(ssim_per_image(o, y).cpu())
    out = {}
    for k, d in acc.items():
        if d["psnr"]:
            out[f"{k}_psnr"] = torch.cat(d["psnr"]).mean().item()
            out[f"{k}_ssim"] = torch.cat(d["ssim"]).mean().item()
    out["latent_drop_db"] = out["full_psnr"] - out["zero_latent_psnr"]
    if "zero_skips_psnr" in out:
        out["skip_drop_db"] = out["full_psnr"] - out["zero_skips_psnr"]
    return out


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ======================================================================================
# Training / evaluation
# ======================================================================================
def make_optimizer_and_scheduler(model, lr, weight_decay, steps_per_epoch, epochs, warmup_epochs=1):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total = max(1, steps_per_epoch * epochs)
    warm = max(1, int(steps_per_epoch * warmup_epochs))

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        t = (step - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, t)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


def make_scaler(device, amp=True):
    """One GradScaler per training run (fp16 AMP on CUDA only; a no-op on CPU)."""
    return torch.amp.GradScaler("cuda", enabled=bool(amp and device.type == "cuda"))


def train_one_epoch(model, loader, loss_fn, optimizer, scheduler, device, scaler=None, amp=True,
                    grad_clip=1.0, max_batches=None) -> dict:
    model.train()
    use_amp = amp and device.type == "cuda"
    if scaler is None:
        scaler = make_scaler(device, amp)
    tot = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0
    for b, batch in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x, y = batch[0].to(device, non_blocking=True), batch[1].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        loss, parts = loss_fn(pred, y)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"non-finite loss at batch {b}")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        bs = x.shape[0]
        tot["loss"] += loss.item() * bs
        tot["l1"] += parts["l1"].item() * bs
        tot["ssim"] += parts["ssim"].item() * bs
        n += bs
    return {k: v / max(n, 1) for k, v in tot.items()}


@torch.no_grad()
def evaluate(model, loader, device, loss_fn=None, amp=True, per_image=False, include_input=False) -> dict:
    """Metrics over a (manifest) loader. Returns means overall and per corruption type.
    per_image=True also returns numpy arrays aligned with the dataset order.
    include_input=True also scores the corrupted input itself (the 'do nothing' baseline)."""
    model.eval()
    use_amp = amp and device.type == "cuda"
    keys = ["psnr", "ssim", "mae"] + (["in_psnr", "in_ssim", "in_mae"] if include_input else [])
    acc = {k: [] for k in keys}
    labels, losses = [], []
    for batch in loader:
        x, y, lab = batch[0].to(device), batch[1].to(device), batch[2]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = model(x)
        pred = pred.float().clamp(0, 1)
        if loss_fn is not None:
            losses.append(loss_fn(pred, y)[0].item() * x.shape[0])
        acc["psnr"].append(psnr_per_image(pred, y).cpu())
        acc["ssim"].append(ssim_per_image(pred, y).cpu())
        acc["mae"].append(mae_per_image(pred, y).cpu())
        if include_input:
            acc["in_psnr"].append(psnr_per_image(x, y).cpu())
            acc["in_ssim"].append(ssim_per_image(x, y).cpu())
            acc["in_mae"].append(mae_per_image(x, y).cpu())
        labels.append(lab.cpu())
    arr = {k: torch.cat(v).numpy() for k, v in acc.items()}
    labels = torch.cat(labels).numpy()
    out = {k: float(v.mean()) for k, v in arr.items()}
    for c, name in enumerate(CLASS_NAMES):
        m = labels == c
        if m.any():
            for k, v in arr.items():
                out[f"{k}/{name}"] = float(v[m].mean())
    out["score"] = val_score(out["psnr"], out["ssim"])
    if loss_fn is not None:
        out["loss"] = sum(losses) / len(labels)
    if per_image:
        out["per_image"] = {**arr, "label": labels}
    return out


@torch.no_grad()
def predict_numpy(model, imgs_hwc: np.ndarray, device, batch_size=64) -> np.ndarray:
    """(N,H,W,3) float [0,1] -> restored (N,H,W,3) float [0,1]."""
    model.eval()
    outs = []
    for i in range(0, len(imgs_hwc), batch_size):
        x = torch.from_numpy(np.ascontiguousarray(imgs_hwc[i:i + batch_size].transpose(0, 3, 1, 2))).float().to(device)
        outs.append(model(x).float().clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1))
    return np.concatenate(outs)


# ======================================================================================
# ONNX
# ======================================================================================
def export_onnx(model, path, img_size=128, opset=17, input_name="input", output_name="output"):
    model = model.eval().cpu()
    dummy = torch.rand(1, 3, img_size, img_size)
    kw = dict(input_names=[input_name], output_names=[output_name], opset_version=opset,
              dynamic_axes={input_name: {0: "batch"}, output_name: {0: "batch"}})
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        try:
            torch.onnx.export(model, dummy, str(path), dynamo=False, **kw)
        except TypeError:          # torch < 2.5 has no `dynamo` kwarg
            torch.onnx.export(model, dummy, str(path), **kw)
    try:
        import onnx
        onnx.checker.check_model(onnx.load(str(path)))
    except ImportError:
        pass
    return path


def verify_onnx(model, path, x: torch.Tensor, atol=1e-4) -> dict:
    """Compare PyTorch (CPU, eval) and onnxruntime outputs on the same batch."""
    import onnxruntime as ort
    model = model.eval().cpu()
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref = model(x.cpu().float()).numpy()
    out = sess.run(None, {sess.get_inputs()[0].name: x.cpu().float().numpy()})[0]
    diff = np.abs(out - ref)
    p_ref = psnr_per_image(torch.from_numpy(ref), torch.from_numpy(out))
    return {"max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean()),
            "min_psnr_torch_vs_onnx": float(p_ref.min()), "n_images": int(len(x)),
            "passed": bool(diff.max() < atol)}


def onnx_latency_ms(path, img_size=128, batch=1, n_warmup=10, n_runs=100) -> dict:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    x = np.random.rand(batch, 3, img_size, img_size).astype(np.float32)
    for _ in range(n_warmup):
        sess.run(None, {name: x})
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {name: x})
        ts.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": float(np.median(ts)), "p90_ms": float(np.percentile(ts, 90)), "batch": batch}


# ======================================================================================
# Plotting
# ======================================================================================
def restoration_grid(rows: list[dict], path=None, title=None, err_vmax=0.5):
    """rows: dicts with 'clean', 'corrupted', 'restored' (HxWx3 float [0,1]) and optional 'label'.
    Columns: clean target | corrupted input | reconstruction | absolute error map."""
    import matplotlib.pyplot as plt
    n = len(rows)
    fig, axes = plt.subplots(n, 4, figsize=(9.2, 2.35 * n), squeeze=False)
    for i, r in enumerate(rows):
        err = np.abs(r["restored"] - r["clean"]).mean(-1)
        panels = [r["clean"], r["corrupted"], r["restored"]]
        for j, im in enumerate(panels):
            axes[i, j].imshow(np.clip(im, 0, 1))
        hm = axes[i, 3].imshow(err, cmap="magma", vmin=0, vmax=err_vmax)
        for ax in axes[i]:
            ax.set_xticks([]); ax.set_yticks([])
        axes[i, 0].set_ylabel(r.get("label", ""), fontsize=7.5, rotation=0, ha="right", va="center", labelpad=4)
        if "in_text" in r:
            axes[i, 1].set_xlabel(r["in_text"], fontsize=7.5)
        if "out_text" in r:
            axes[i, 2].set_xlabel(r["out_text"], fontsize=7.5)
    for j, t in enumerate(["clean target", "corrupted input", "reconstruction", "|error| (mean RGB)"]):
        axes[0, j].set_title(t, fontsize=9)
    fig.colorbar(hm, ax=axes[:, 3].tolist(), fraction=0.046, pad=0.02, shrink=0.6)
    if title:
        fig.suptitle(title, fontsize=11)
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    return fig

In [ ]:
import os, sys, json, time, math, shutil, importlib, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import optuna

sys.path.insert(0, str(Path.cwd()))
import pet_data as pdt, restoration as R
importlib.reload(pdt); importlib.reload(R)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================ MAIN SWITCH ============================
MODE = "smoke"          # "smoke" first, then "full"
# =====================================================================
MODE = os.environ.get("T1_MODE", MODE)

SETTINGS = {
    #            subsets (None = all)                    Optuna                          final training         ablation
    "smoke": dict(train_subset=512, val_per_class=32, test_images=40, n_trials=3, trial_epochs=2, optuna_timeout=None,
                  final_epochs=5, patience=99, sample_every=1, ablation_epochs=1, overfit_steps=300, overfit_per_class=4),
    "full":  dict(train_subset=None, val_per_class=None, test_images=None, n_trials=30, trial_epochs=12, optuna_timeout=5 * 3600,
                  final_epochs=80, patience=15, sample_every=5, ablation_epochs=20, overfit_steps=300, overfit_per_class=4),
}
S = dict(SETTINGS[MODE])
S.update(json.loads(os.environ.get("T1_SETTINGS_OVERRIDE", "{}")))   # local CPU testing only
RUN_ABLATION = True

IS_KAGGLE  = Path("/kaggle/input").exists()
INPUT_ROOT = Path(os.environ.get("PET_INPUT_ROOT", "/kaggle/input"))
WORK_DIR   = Path(os.environ.get("WORK_DIR", "/kaggle/working" if IS_KAGGLE else "./work")).resolve()
IMAGES_DIR, ANNOT_DIR = None, None      # set manually only if auto-discovery fails

TAG = "task1" if MODE == "full" else "task1_smoke"
DIRS = {k: WORK_DIR / k / TAG for k in ["checkpoints", "models", "figures", "results", "optuna"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
WEIGHT_DECAY = 1e-4          # fixed (not in the required search space)
AMP = True                   # fp16 mixed precision on CUDA
NUM_WORKERS = pdt.default_num_workers()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pdt.seed_everything(SEED)

USE_WANDB = True
WANDB_PROJECT = "genai-assignment1"
WANDB_ENTITY = None
WANDB_SECRET_NAME = "WANDB_API_KEY"
LOG_TRIALS_TO_WANDB = True   # one W&B run per Optuna trial
GROUP = "task1-v2" if MODE == "full" else "task1-v2-smoke"

print(f"MODE={MODE} | device={device} {torch.cuda.get_device_name(0) if device.type == 'cuda' else ''} | "
      f"torch {torch.__version__} | optuna {optuna.__version__} | workers={NUM_WORKERS}")
print(json.dumps(S))

In [ ]:
WANDB_OK = False
if USE_WANDB:
    try:
        import wandb
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            wandb.login(key=UserSecretsClient().get_secret(WANDB_SECRET_NAME))
        WANDB_OK = True
    except Exception as e:
        print("W&B disabled:", repr(e))

def wb_init(name, job_type, config=None, **kw):
    if not WANDB_OK:
        return None
    if wandb.run is not None:
        wandb.finish()
    return wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group=GROUP, name=name, job_type=job_type,
                      tags=["task1", "v2", MODE], config=config or {}, **kw)

def wb_log_figs(run, figs: dict):
    if run is not None:
        run.log({k: wandb.Image(str(v)) for k, v in figs.items()})

print("W&B:", "on" if WANDB_OK else "off")
if USE_WANDB and not WANDB_OK:
    raise RuntimeError(
        "W&B login failed, so nothing would be logged. Fix before training (W&B logging is required):\n"
        "  1. Add-ons -> Secrets -> tick WANDB_API_KEY for THIS notebook (secrets are attached per notebook)\n"
        "  2. Settings -> Internet -> On\n"
        "Set USE_WANDB = False only if you deliberately want to run without logging.")

## 1. Data

In [ ]:
data = pdt.prepare_data(INPUT_ROOT, WORK_DIR, IMAGES_DIR, ANNOT_DIR)

train_images = data.train_images if S["train_subset"] is None else data.train_images[:S["train_subset"]]
val_records = data.val_manifest["records"]
if S["val_per_class"] is not None:
    val_records = [r for c in range(4) for r in [x for x in val_records if x["label"] == c][:S["val_per_class"]]]
val_loader = pdt.make_eval_loader(data.val_images, data.val_name_to_row, val_records, batch_size=64, num_workers=NUM_WORKERS)

# Fixed validation images for progress grids (same images every time -> training can be followed visually)
PROGRESS_RECORDS = [r for c in range(4) for r in [x for x in val_records if x["label"] == c][:2]]
print(f"\ntrain images {len(train_images)} | val records {len(val_records)} "
      f"({pd.Series([pdt.CLASS_NAMES[r['label']] for r in val_records]).value_counts().to_dict()})")

### Pre-flight gate
Everything needed for the full run is verified here, before any training. The notebook stops with instructions if a check fails.

In [ ]:
pre = []
def pf(name, ok, detail="", fatal=True):
    pre.append({"check": name, "passed": bool(ok), "fatal": fatal, "detail": str(detail)})
    print(("PASS " if ok else ("FAIL " if fatal else "WARN ")) + name + (f"  — {detail}" if detail else ""))

# 1. GPU (required in full mode)
gpu = torch.cuda.is_available()
pf("GPU available", gpu, torch.cuda.get_device_name(0) if gpu else "Settings -> Accelerator -> GPU T4", fatal=(MODE == "full"))

# 2. W&B end-to-end: create a real run, log a value, finish it
wb_ok, wb_detail = False, "USE_WANDB is False"
if USE_WANDB:
    try:
        r = wb_init("preflight", "preflight", {"mode": MODE, "settings": S})
        r.log({"preflight/ok": 1}); wb_detail = r.url; r.finish(); wb_ok = True
    except Exception as e:
        wb_detail = repr(e)
pf("W&B login + test run", wb_ok or not USE_WANDB, wb_detail)

# 3. Data
pf("dataset found (3680 trainval / 3669 test names)", len(data.split["train"]) + len(data.split["val"]) >= 3600 and len(data.test_names) >= 3600,
   f"{len(data.split['train'])} train / {len(data.split['val'])} val / {len(data.test_names)} test")
pf("notebook-00 output attached (same split + manifests as Tasks 2-3)", data.source.startswith("attached"), data.source, fatal=False)
pf("split / manifest hashes recorded", all(data.hashes.values()), {k: v[:12] for k, v in data.hashes.items()})

# 4. Libraries, storage, disk
try:
    import onnx, onnxruntime
    pf("onnx + onnxruntime importable", True, f"onnx {onnx.__version__}, ort {onnxruntime.__version__}")
except Exception as e:
    pf("onnx + onnxruntime importable", False, repr(e))
ok = True
for d in DIRS.values():
    try:
        (d / ".probe").write_text("x"); (d / ".probe").unlink()
    except Exception:
        ok = False
pf("output folders writeable", ok, str(WORK_DIR))
free = shutil.disk_usage(WORK_DIR).free / 1e9
pf("free disk > 5 GB", free > 5, f"{free:.1f} GB")
if MODE == "full":
    pf("Optuna study starts fresh", not (DIRS["optuna"] / "task1_universal_dae.db").exists(),
       "an old study DB exists in /kaggle/working — it would be resumed", fatal=False)

pre_df = pd.DataFrame(pre)
bad = pre_df[pre_df.fatal & ~pre_df.passed]
assert bad.empty, "PRE-FLIGHT FAILED — nothing has been trained. Fix these and re-run:\n" + bad.to_string()
print(f"\nPRE-FLIGHT OK ({int(pre_df.passed.sum())}/{len(pre_df)} passed) — safe to train.")

In [ ]:
def records_to_arrays(records, images, name_to_row):
    clean = np.stack([images[name_to_row[r["image"]]].astype(np.float32) / 255 for r in records])
    corrupted = np.stack([pdt.apply_corruption(c, r) for c, r in zip(clean, records)])
    return corrupted, clean

def progress_figure(model, path, title):
    cor, cl = records_to_arrays(PROGRESS_RECORDS, data.val_images, data.val_name_to_row)
    out = R.predict_numpy(model, cor, device)
    rows = [dict(clean=cl[i], corrupted=cor[i], restored=out[i], label=pdt.describe_params(r))
            for i, r in enumerate(PROGRESS_RECORDS)]
    fig = R.restoration_grid(rows, path, title); plt.close(fig)
    return path

## 2. Unit tests and the overfit-one-batch test
These run in both modes and catch shape, gradient, loss, metric and export bugs **before** any long training starts.

In [ ]:
tests = []
def check(name, ok, detail=""):
    tests.append({"test": name, "passed": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + (f"  — {detail}" if detail else ""))

x = torch.rand(4, 3, 128, 128, device=device)
for s, sc in ((0, 4), (1, 4), (2, 2)):
    m = R.ConvAutoencoder(32, 16, 0.1, s, sc).to(device).eval()
    with torch.no_grad():
        y = m(x); z, skips = m.encode(x)
    sent = z[0].numel() + sum(t[0].numel() for t in skips)       # values the decoder receives per image
    check(f"skip_levels={s}: output (4,3,128,128) in [0,1]", tuple(y.shape) == (4, 3, 128, 128) and 0 <= y.min() and y.max() <= 1)
    check(f"skip_levels={s}, skip_channels={sc}: decoder receives exactly code_dim={m.code_dim} values ({m.compression_ratio:.1f}x compression)",
          sent == m.code_dim and m.code_dim <= 3 * 128 * 128 // 8, f"latent {m.latent_dim} + skips {m.skip_dim}; {R.count_params(m) / 1e6:.2f}M params")
wide = R.ConvAutoencoder(32, 16, 0.1, 2, None)
check("wide-skip (v1) design is flagged as NOT compressed", wide.compression_ratio < 1, f"code {wide.code_dim} values = {wide.compression_ratio:.2f}x")
del wide
m0 = R.ConvAutoencoder(32, 32, 0.0, 0).to(device).eval()
with torch.no_grad():
    z, skips = m0.encode(x)
    check("pure AE: output depends only on the latent", skips == [] and torch.allclose(m0.decode(z, None), m0(x)))
    x2 = x.clone(); x2[:, :, :64] = 0           # change input: output must change through the latent
    d = (m0(x2) - m0(x)).abs().max().item()
    check("pure AE: different input -> different output", d > 1e-3, f"max |Δout| = {d:.3f}")

m = R.ConvAutoencoder(32, 32, 0.1, 2).to(device).train()
loss, parts = R.L1SSIMLoss(0.8)(m(x), torch.rand_like(x))
loss.backward()
no_grad = [n for n, p in m.named_parameters() if p.grad is None or not torch.isfinite(p.grad).all()]
check("gradients reach every parameter and are finite", not no_grad, no_grad[:3])

a = torch.rand(2, 3, 128, 128); b = (a + 0.1 * torch.randn_like(a)).clamp(0, 1)
check("L1SSIM loss of identical images = 0", abs(R.L1SSIMLoss(0.8)(a, a)[0].item()) < 1e-6)
check("alpha=1 -> pure L1", abs(R.L1SSIMLoss(1.0)(a, b)[0].item() - (a - b).abs().mean().item()) < 1e-6)
try:
    from skimage.metrics import structural_similarity as sk_ssim
    ref = np.array([sk_ssim(a[i].permute(1, 2, 0).numpy(), b[i].permute(1, 2, 0).numpy(), channel_axis=-1, data_range=1,
                            gaussian_weights=True, sigma=1.5, use_sample_covariance=False) for i in range(2)])
    d = np.abs(R.ssim_per_image(a, b).numpy() - ref).max()
    check("SSIM matches scikit-image", d < 1e-4, f"max |diff| = {d:.1e}")
except ImportError:
    print("skimage not installed — SSIM cross-check skipped")
mse = ((a - b) ** 2).mean().item()
check("PSNR formula", abs(R.psnr_per_image(a[:1], b[:1]).item() - 10 * math.log10(1 / ((a[:1] - b[:1]) ** 2).mean().item())) < 1e-3)

cd = R.code_dependence(R.ConvAutoencoder(32, 16, 0.0, 2, 2).to(device), val_loader, device, max_batches=1)
check("code_dependence() runs (full / zero-latent / zero-skips)", {"latent_drop_db", "skip_drop_db"} <= set(cd), {k: round(v, 2) for k, v in cd.items()})
ev = R.evaluate(R.ConvAutoencoder(32, 32).to(device), val_loader, device, R.L1SSIMLoss(0.8))
check("evaluate() returns overall + per-type metrics", all(f"psnr/{n}" in ev for n in pdt.CLASS_NAMES), f"score={ev['score']:.3f}")
probe = DIRS["models"] / "_probe.onnx"
m_cpu = R.ConvAutoencoder(32, 32, 0.1, 2)
R.export_onnx(m_cpu, probe)
par = R.verify_onnx(m_cpu, probe, torch.rand(3, 3, 128, 128))
check("untrained model exports to ONNX and matches PyTorch", par["passed"], f"max |diff| = {par['max_abs_diff']:.1e}")
probe.unlink()
del m, m0, m_cpu

In [ ]:
# Overfit one fixed batch — the classic wiring check.
# (a) pure AE: must escape the initial loss and reach the "blurry mean image" plateau (proves gradients flow
#     through the bottleneck path). Measured: ~47% of the initial loss after ~50 steps, so < 70% is a safe bound.
# (b) skip model: must memorise the batch (proves the network can fit targets). Reported as a warning only,
#     because the pure AE's plateau can legitimately last several hundred steps.
fixed = [r for c in range(4) for r in [x for x in val_records if x["label"] == c][:S["overfit_per_class"]]]
cor, cl = records_to_arrays(fixed, data.val_images, data.val_name_to_row)
xb = torch.from_numpy(cor.transpose(0, 3, 1, 2)).to(device); yb = torch.from_numpy(cl.transpose(0, 3, 1, 2)).to(device)
loss_fn = R.L1SSIMLoss(0.8)
curves = {}
for name, skips, lr in [("pure AE (skip_levels=0)", 0, 1e-3), ("bottlenecked skips (skip_levels=2, 4 ch)", 2, 2e-3)]:
    torch.manual_seed(SEED)
    model = R.ConvAutoencoder(32, 32, 0.0, skips, 4).to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    curve = []
    for step in range(S["overfit_steps"]):
        loss, _ = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        curve.append(loss.item())
    model.eval()
    with torch.no_grad():
        p = R.psnr_per_image(model(xb), yb).mean().item()
    curves[name] = (curve, p)
    del model, opt
(c0, p0), (c2, p2) = curves.values()
check("overfit: pure AE learns (final loss < 70% of initial)", c0[-1] < 0.7 * c0[0], f"{c0[0]:.4f} -> {c0[-1]:.4f}, PSNR {p0:.1f} dB")
if c2[-1] < 0.35 * c2[0]:
    print(f"PASS overfit: skip model memorises the batch  — {c2[0]:.4f} -> {c2[-1]:.4f}, PSNR {p2:.1f} dB")
else:
    print(f"WARN overfit: skip model only reached {c2[-1] / c2[0]:.0%} of the initial loss (PSNR {p2:.1f} dB) — "
          "not a failure, but mention it if training later looks stuck")
plt.figure(figsize=(6, 3.2))
for name, (c, p) in curves.items():
    plt.plot(c, label=f"{name}  ({p:.1f} dB)")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("L1+SSIM loss"); plt.legend(fontsize=8)
plt.title("Overfit-one-batch sanity test"); plt.tight_layout(); plt.savefig(DIRS["figures"] / "overfit_one_batch.png", dpi=130); plt.show()

tests_df = pd.DataFrame(tests)
assert tests_df.passed.all(), f"UNIT TESTS FAILED — fix before training:\n{tests_df[~tests_df.passed]}"
print(f"\nall {len(tests_df)} tests passed")

## 3. Optuna hyper-parameter search
* **Objective (maximised):** $0.5\cdot\mathrm{SSIM}_{val} + 0.5\cdot\min(\mathrm{PSNR}_{val}/40,\,1)$ on the fixed validation manifest (all four conditions).
  It does not depend on α, so trials with a different loss weighting are compared on equal terms.
* **Sampler:** TPE (seed 42, multivariate). **Pruner:** median pruner after 4 warm-up epochs.
* **Storage:** SQLite. The study resumes automatically if a session dies, and the `.db` file goes into the repo.
* The first trial is the hand-picked baseline (α = 0.8, …) so the tuned configuration can be compared against it.
* **Genuine-bottleneck rule:** a configuration is valid only if the total code it passes to the decoder (latent + bottlenecked skips) is at most 1/8 of the input, i.e. 6,144 values.
  Invalid configurations are rejected before training, recorded as `invalid`, and not counted towards the trial budget.

In [ ]:
SEARCH_SPACE = {
    "lr":              {"type": "float", "low": 1e-4, "high": 3e-3, "log": True},
    "batch_size":      {"type": "categorical", "choices": [16, 32, 64]},
    "latent_channels": {"type": "categorical", "choices": [8, 16, 32, 64]},     # bottleneck = ch x 8 x 8 = 512..4096 values
    "base_channels":   {"type": "categorical", "choices": [32, 48, 64]},        # encoder widths base x (1,2,4,8,8)
    "dropout":         {"type": "float", "low": 0.0, "high": 0.3},
    "alpha":           {"type": "float", "low": 0.5, "high": 0.95},              # L1 weight
    "skip_levels":     {"type": "categorical", "choices": [0, 1, 2]},           # bottlenecked skips at 16x16 / 32x32
    "skip_channels":   {"type": "categorical", "choices": [2, 4, 8]},           # width of each skip code (ignored if skip_levels=0)
}
MAX_CODE = 3 * 128 * 128 // 8          # 6,144 values: total code must compress the input at least 8x
BASELINE = {"lr": 1e-3, "batch_size": 32, "latent_channels": 32, "base_channels": 32, "dropout": 0.1, "alpha": 0.8,
            "skip_levels": 0, "skip_channels": 4}
pdt.save_json(SEARCH_SPACE, DIRS["results"] / "search_space.json", indent=1)

def suggest(trial):
    hp = {}
    for k, s in SEARCH_SPACE.items():
        if s["type"] == "float":
            hp[k] = trial.suggest_float(k, s["low"], s["high"], log=s.get("log", False))
        else:
            hp[k] = trial.suggest_categorical(k, s["choices"])
    return hp

def build_model(hp):
    return R.ConvAutoencoder(hp["base_channels"], hp["latent_channels"], hp["dropout"], hp["skip_levels"],
                             hp.get("skip_channels", 4)).to(device)

def objective(trial):
    hp = suggest(trial)
    probe = R.ConvAutoencoder(8, hp["latent_channels"], 0, hp["skip_levels"], hp["skip_channels"])   # cheap size check
    trial.set_user_attr("code_dim", probe.code_dim); trial.set_user_attr("compression", probe.compression_ratio)
    if probe.code_dim > MAX_CODE:
        trial.set_user_attr("invalid", f"code {probe.code_dim} > {MAX_CODE}")
        raise optuna.TrialPruned(f"invalid: code_dim {probe.code_dim} > {MAX_CODE}")
    torch.manual_seed(SEED + trial.number)
    model = build_model(hp)
    loader = pdt.make_train_loader(train_images, batch_size=hp["batch_size"], num_workers=NUM_WORKERS)
    loss_fn = R.L1SSIMLoss(hp["alpha"])
    E = S["trial_epochs"]
    opt, sch = R.make_optimizer_and_scheduler(model, hp["lr"], WEIGHT_DECAY, len(loader), E)
    scaler = R.make_scaler(device, AMP)
    info = {"latent_dim": model.latent_dim, "code_dim": model.code_dim, "compression": model.compression_ratio,
            "params_M": R.count_params(model) / 1e6}
    trial.set_user_attr("latent_dim", model.latent_dim); trial.set_user_attr("params_M", info["params_M"])
    run = wb_init(f"trial-{trial.number:03d}", "optuna-trial", {**hp, **info, "trial": trial.number, "epochs": E}) \
        if LOG_TRIALS_TO_WANDB else None
    best, state = -1.0, "complete"
    try:
        for ep in range(E):
            t0 = time.time()
            tr = R.train_one_epoch(model, loader, loss_fn, opt, sch, device, scaler, AMP)
            ev = R.evaluate(model, val_loader, device, loss_fn, AMP)
            if ev["score"] > best:
                best = ev["score"]
                for k in ("psnr", "ssim", "mae"):
                    trial.set_user_attr(f"val_{k}", ev[k])
                trial.set_user_attr("best_epoch", ep)
            if run:
                run.log({"epoch": ep, "train/loss": tr["loss"], "train/l1": tr["l1"], "train/ssim": tr["ssim"],
                         "val/loss": ev["loss"], "val/score": ev["score"], "val/psnr": ev["psnr"], "val/ssim": ev["ssim"],
                         **{f"val/{k}": v for k, v in ev.items() if "/" in k}, "epoch_time_s": time.time() - t0})
            trial.report(ev["score"], ep)
            if trial.should_prune():
                state = "pruned"
                raise optuna.TrialPruned()
    except torch.cuda.OutOfMemoryError:
        state = "oom"
        raise optuna.TrialPruned("CUDA OOM")
    finally:
        if run:
            run.summary.update({"best_score": best, "state": state}); run.finish()
        del model, loader, opt
        gc.collect(); torch.cuda.empty_cache()
    return best

In [ ]:
STUDY_NAME = f"{TAG}_universal_dae"
DB_PATH = DIRS["optuna"] / f"{STUDY_NAME}.db"
if MODE == "smoke" and DB_PATH.exists():
    DB_PATH.unlink()                                   # smoke runs always start fresh

study = optuna.create_study(
    study_name=STUDY_NAME, storage=f"sqlite:///{DB_PATH}", direction="maximize", load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True, n_startup_trials=min(8, S["n_trials"])),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=4))
if len(study.trials) == 0:
    study.enqueue_trial(BASELINE)

def n_valid():
    return len([t for t in study.trials if "invalid" not in t.user_attrs and
                t.state in (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)])

print(f"study '{STUDY_NAME}': {n_valid()} valid finished trials, target {S['n_trials']}")
t_study, attempts = time.time(), 0
while n_valid() < S["n_trials"] and attempts < 6 * S["n_trials"]:
    left = None if S["optuna_timeout"] is None else S["optuna_timeout"] - (time.time() - t_study)
    if left is not None and left <= 0:
        print("Optuna time budget reached"); break
    study.optimize(objective, n_trials=1, timeout=left, gc_after_trial=True, catch=(FloatingPointError,))
    attempts += 1
n_invalid = len([t for t in study.trials if "invalid" in t.user_attrs])
print(f"study time {(time.time() - t_study) / 60:.1f} min | valid trials {n_valid()} | rejected (code too large) {n_invalid}")

In [ ]:
trials_df = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration"))
trials_df.to_csv(DIRS["results"] / "optuna_trials.csv", index=False)
states = trials_df["state"].value_counts().to_dict()
if "user_attrs_invalid" in trials_df:
    states["INVALID (rejected, not trained)"] = int(trials_df["user_attrs_invalid"].notna().sum())
    states["PRUNED"] = states.get("PRUNED", 0) - states["INVALID (rejected, not trained)"]
best_trial = study.best_trial
BEST_HP = dict(best_trial.params)
baseline_row = trials_df[trials_df.number == 0]
print("trial states:", states)
print(f"best trial #{best_trial.number}: score={best_trial.value:.4f}  "
      f"PSNR={best_trial.user_attrs.get('val_psnr', float('nan')):.2f}  SSIM={best_trial.user_attrs.get('val_ssim', float('nan')):.4f}")
print("best params:", json.dumps(BEST_HP, indent=1))
print(f"best code: {best_trial.user_attrs.get('code_dim')} values ({best_trial.user_attrs.get('compression', 0):.1f}x compression)")
if len(baseline_row):
    print(f"baseline (trial 0) score={baseline_row['value'].iloc[0]:.4f}")
display(trials_df.sort_values("value", ascending=False).head(10))
plot_study = optuna.create_study(direction="maximize")          # plots only on trained trials (invalid ones carry no value)
plot_study.add_trials([t for t in study.trials if "invalid" not in t.user_attrs])

optuna_figs = {}
from optuna.visualization import matplotlib as ovm
for name, fn in [("history", ovm.plot_optimization_history), ("importance", ovm.plot_param_importances),
                 ("slice", ovm.plot_slice), ("parallel", ovm.plot_parallel_coordinate)]:
    try:
        obj = fn(plot_study)
        fig = (obj.flat[0] if hasattr(obj, "flat") else obj).figure
        fig.set_size_inches(12 if name in ("slice", "parallel") else 7, 4.5)
        p = DIRS["figures"] / f"optuna_{name}.png"
        fig.savefig(p, dpi=140, bbox_inches="tight"); plt.show(); optuna_figs[f"optuna/{name}"] = p
    except Exception as e:
        print(f"optuna plot '{name}' skipped: {e}")

pdt.save_json({"best_trial": best_trial.number, "best_value": best_trial.value, "best_params": BEST_HP,
               "user_attrs": best_trial.user_attrs, "n_trials": len(study.trials), "states": states,
               "trial_epochs": S["trial_epochs"], "search_space": SEARCH_SPACE, "baseline": BASELINE, "max_code": MAX_CODE},
              DIRS["results"] / "optuna_best.json", indent=1)

run = wb_init("optuna-summary", "optuna-summary", {"search_space": SEARCH_SPACE, "n_trials": len(study.trials)})
if run:
    run.summary.update({"best_value": best_trial.value, "best_trial": best_trial.number, **{f"best/{k}": v for k, v in BEST_HP.items()}})
    run.log({"optuna_trials": wandb.Table(dataframe=trials_df.astype(str))})
    wb_log_figs(run, optuna_figs)
    art = wandb.Artifact(f"{STUDY_NAME}-study", type="optuna-study"); art.add_file(str(DB_PATH)); art.add_file(str(DIRS["results"] / "optuna_trials.csv"))
    run.log_artifact(art); run.finish()

## 4. Final training with the selected configuration

In [ ]:
FINAL_HP = dict(BEST_HP)          # override here only if you have a documented reason
if S.get("final_batch_size"):
    FINAL_HP["batch_size"] = S["final_batch_size"]       # local CPU testing only
FINAL_EPOCHS, PATIENCE = S["final_epochs"], S["patience"]
pdt.save_json(FINAL_HP, DIRS["results"] / "final_hp.json", indent=1)

torch.manual_seed(SEED)
model = build_model(FINAL_HP)
train_loader = pdt.make_train_loader(train_images, batch_size=FINAL_HP["batch_size"], num_workers=NUM_WORKERS)
loss_fn = R.L1SSIMLoss(FINAL_HP["alpha"])
opt, sch = R.make_optimizer_and_scheduler(model, FINAL_HP["lr"], WEIGHT_DECAY, len(train_loader), FINAL_EPOCHS, warmup_epochs=2)
scaler = R.make_scaler(device, AMP)
BEST_CKPT, LAST_CKPT = DIRS["checkpoints"] / "task1_best.pth", DIRS["checkpoints"] / "task1_last.pth"
model_info = {"latent_shape": model.latent_shape, "latent_dim": model.latent_dim, "skip_shapes": model.skip_shapes,
              "skip_dim": model.skip_dim, "code_dim": model.code_dim, "compression_ratio": model.compression_ratio,
              "params_M": R.count_params(model) / 1e6}
print(json.dumps(model_info)); print(model)

run = wb_init("final-training", "train", {**FINAL_HP, **model_info, "epochs": FINAL_EPOCHS, "patience": PATIENCE,
                                          "weight_decay": WEIGHT_DECAY, "amp": AMP, "n_train": len(train_images),
                                          "n_val_records": len(val_records), "data": data.summary()})
if run:
    run.watch(model, log="gradients", log_freq=200)

history, best_score, best_epoch, start_epoch = [], -1.0, -1, 0
ev_untrained = R.evaluate(model, val_loader, device, loss_fn, AMP)     # reference for the learning check
print(f"untrained: val loss {ev_untrained['loss']:.4f} PSNR {ev_untrained['psnr']:.2f}")
if MODE == "full" and LAST_CKPT.exists():                  # resume after an interrupted session
    ck = torch.load(LAST_CKPT, map_location=device, weights_only=False)
    if ck["hp"] == FINAL_HP:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"])
        history, best_score, best_epoch, start_epoch = ck["history"], ck["best_score"], ck["best_epoch"], ck["epoch"] + 1
        print(f"resumed from epoch {start_epoch}")

t_train = time.time()
for ep in range(start_epoch, FINAL_EPOCHS):
    t0 = time.time()
    tr = R.train_one_epoch(model, train_loader, loss_fn, opt, sch, device, scaler, AMP)
    ev = R.evaluate(model, val_loader, device, loss_fn, AMP)
    rec = {"epoch": ep, "lr": sch.get_last_lr()[0], "time_s": time.time() - t0,
           **{f"train/{k}": v for k, v in tr.items()},
           **{f"val/{k}": v for k, v in ev.items() if k != "per_image"}}
    history.append(rec)
    improved = ev["score"] > best_score
    if improved:
        best_score, best_epoch = ev["score"], ep
        torch.save({"model": model.state_dict(), "model_config": model.config, "hp": FINAL_HP, "epoch": ep,
                    "val": {k: v for k, v in ev.items() if k != "per_image"}}, BEST_CKPT)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sch.state_dict(), "hp": FINAL_HP,
                "epoch": ep, "history": history, "best_score": best_score, "best_epoch": best_epoch}, LAST_CKPT)
    logs = {k: v for k, v in rec.items()}
    if ep % S["sample_every"] == 0 or ep == FINAL_EPOCHS - 1:
        p = progress_figure(model, DIRS["figures"] / "progress_latest.png", f"Validation samples — epoch {ep}")
        if run:
            logs["val_samples"] = wandb.Image(str(p), caption=f"epoch {ep}")
    if run:
        run.log(logs)
    print(f"ep {ep:3d} | train {tr['loss']:.4f} | val {ev['loss']:.4f} PSNR {ev['psnr']:.2f} SSIM {ev['ssim']:.4f} "
          f"score {ev['score']:.4f}{' *' if improved else ''} | {rec['time_s']:.0f}s")
    if ep == start_epoch:
        print(f"   ≈ {rec['time_s'] * (FINAL_EPOCHS - ep) / 60:.0f} min for the remaining epochs (upper bound)")
    if ep - best_epoch >= PATIENCE:
        print(f"early stopping: no improvement for {PATIENCE} epochs"); break
print(f"training {(time.time() - t_train) / 60:.1f} min | best epoch {best_epoch} score {best_score:.4f}")
del train_loader; gc.collect()          # release the persistent data-loader workers
best_val_loss = min(h["val/loss"] for h in history)
learned = best_val_loss < 0.8 * ev_untrained["loss"]
print(f"learning check: best val loss {best_val_loss:.4f} vs untrained {ev_untrained['loss']:.4f} -> {'OK' if learned else 'NOT LEARNING'}")
assert learned, "Validation loss did not improve on the untrained model — inspect before running MODE='full'."

In [ ]:
hist = pd.DataFrame(history); hist.to_csv(DIRS["results"] / "training_history.csv", index=False)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(hist.epoch, hist["train/loss"], label="train"); ax[0].plot(hist.epoch, hist["val/loss"], label="val")
ax[0].set_title(f"L1+SSIM loss (α={FINAL_HP['alpha']:.2f})"); ax[0].legend()
for n in pdt.CLASS_NAMES:
    ax[1].plot(hist.epoch, hist[f"val/psnr/{n}"], label=n); ax[2].plot(hist.epoch, hist[f"val/ssim/{n}"], label=n)
ax[1].plot(hist.epoch, hist["val/psnr"], "k--", label="all"); ax[2].plot(hist.epoch, hist["val/ssim"], "k--", label="all")
ax[1].set_title("validation PSNR (dB)"); ax[2].set_title("validation SSIM"); ax[2].legend(fontsize=8)
for a in ax:
    a.axvline(best_epoch, color="grey", ls=":", lw=1); a.set_xlabel("epoch")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "training_curves.png", dpi=150); plt.show()

ck = torch.load(BEST_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ck["model"]); model.eval()
print(f"loaded best checkpoint (epoch {ck['epoch']}) for evaluation")

# Is the latent genuinely used? Zero it (and, separately, the skip codes) and measure the damage on validation.
dep = R.code_dependence(model, val_loader, device)
print(json.dumps({k: round(v, 3) for k, v in dep.items()}, indent=1))
if dep["latent_drop_db"] < 3:
    print(f"WARNING: zeroing the latent costs only {dep['latent_drop_db']:.1f} dB — the decoder barely uses it. Discuss in the report.")
else:
    print(f"latent is essential: zeroing it costs {dep['latent_drop_db']:.1f} dB")
pdt.save_json(dep, DIRS["results"] / "code_dependence_final.json", indent=1)
if run:
    run.summary.update({f"code_dependence/{k}": v for k, v in dep.items()})
    wb_log_figs(run, {"figures/training_curves": DIRS["figures"] / "training_curves.png"})
    art = wandb.Artifact("task1-universal-dae-checkpoint", type="model", metadata={**FINAL_HP, **model_info, "best_epoch": best_epoch})
    art.add_file(str(BEST_CKPT)); run.log_artifact(art)

## 5. Test-set evaluation (by corruption type and severity)
The official test set is used here for the first time. Each image is evaluated under 10 fixed conditions from the test manifest. The *input* columns score the corrupted image itself, i.e. the do-nothing baseline.

In [ ]:
test_images, test_row, test_failed = data.load_test_images()
test_records = [r for r in data.test_manifest["records"] if r["image"] in test_row]
if S["test_images"] is not None:
    keep = set(data.test_names[:S["test_images"]]); test_records = [r for r in test_records if r["image"] in keep]
test_loader = pdt.make_eval_loader(test_images, test_row, test_records, batch_size=128, num_workers=NUM_WORKERS)
t0 = time.time()
ev_test = R.evaluate(model, test_loader, device, None, AMP, per_image=True, include_input=True)
print(f"{len(test_records)} test records in {time.time() - t0:.1f}s (unreadable test images: {test_failed})")

pi = ev_test["per_image"]
tdf = pd.DataFrame([{k: r.get(k) for k in ("id", "image", "type", "severity_level", "p", "kernel_size", "sigma", "n_boxes", "coverage")}
                    for r in test_records])
for k in ("psnr", "ssim", "mae", "in_psnr", "in_ssim", "in_mae"):
    tdf[k] = pi[k]
tdf["d_psnr"], tdf["d_ssim"] = tdf.psnr - tdf.in_psnr, tdf.ssim - tdf.in_ssim
# A clean input equals its target, so its PSNR is infinite (the metric caps it at 100 dB). Leave it blank rather than
# letting 100 dB distort every average that includes clean images.
tdf.loc[tdf.type == "clean", ["in_psnr", "d_psnr"]] = np.nan
tdf.to_csv(DIRS["results"] / "test_per_record.csv", index=False)

cols = {"in_psnr": "PSNR in", "psnr": "PSNR out", "d_psnr": "ΔPSNR", "in_ssim": "SSIM in", "ssim": "SSIM out",
        "d_ssim": "ΔSSIM", "mae": "MAE out"}
order = ["clean", "salt_pepper", "gaussian_blur", "occlusion"]
by_type = tdf.groupby("type")[list(cols)].mean().reindex(order).rename(columns=cols)
by_type.loc["all corrupted"] = tdf[tdf.type != "clean"][list(cols)].mean().rename(cols)
by_type.loc["overall"] = tdf[list(cols)].mean().rename(cols)
by_type.loc["overall", ["PSNR in", "ΔPSNR"]] = np.nan          # undefined: clean inputs have infinite PSNR
sev_order = ["none", "low", "medium", "high"]
by_sev = (tdf.groupby(["type", "severity_level"])[list(cols)].mean().rename(columns=cols)
          .reindex([(t, s) for t in order for s in sev_order if (t, s) in set(zip(tdf.type, tdf.severity_level))]))
print("Results by input condition (clean 'PSNR in' is infinite and left blank; use 'all corrupted' as the headline)")
display(by_type.round(4))
print("Results by condition × severity"); display(by_sev.round(4))
by_type.to_csv(DIRS["results"] / "test_by_type.csv"); by_sev.to_csv(DIRS["results"] / "test_by_type_severity.csv")
for name, t in [("test_by_type", by_type), ("test_by_type_severity", by_sev)]:
    try:
        (DIRS["results"] / f"{name}.tex").write_text(t.round(3).to_latex(float_format="%.3f"))
    except Exception as e:
        print("LaTeX export skipped:", e)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
sev = tdf[tdf.type != "clean"].groupby(["type", "severity_level"])[["psnr", "in_psnr", "ssim", "in_ssim"]].mean()
for i, (m, lab) in enumerate([("psnr", "PSNR (dB)"), ("ssim", "SSIM")]):
    for t, c in zip(order[1:], ["C0", "C1", "C2"]):
        ax[i].plot(["low", "medium", "high"], [sev.loc[(t, s), m] for s in ("low", "medium", "high")], "-o", color=c, label=f"{t} restored")
        ax[i].plot(["low", "medium", "high"], [sev.loc[(t, s), f"in_{m}"] for s in ("low", "medium", "high")], ":x", color=c, label=f"{t} input")
    ax[i].axhline(tdf[tdf.type == "clean"][m].mean(), color="k", lw=1, ls="--", label="clean → restored")
    ax[i].set_ylabel(lab); ax[i].set_xlabel("severity")
ax[1].legend(fontsize=7, ncol=2); fig.suptitle("Test performance vs severity (solid = restored, dotted = corrupted input)")
fig.tight_layout(); fig.savefig(DIRS["figures"] / "test_severity_curves.png", dpi=150); plt.show()

## 6. Representative examples (12) and failure cases (4)

In [ ]:
rec_by_id = {r["id"]: r for r in test_records}

def rows_for(ids):
    recs = [rec_by_id[i] for i in ids]
    cor, cl = records_to_arrays(recs, test_images, test_row)
    out = R.predict_numpy(model, cor, device)
    rows = []
    for i, r in enumerate(recs):
        m = tdf.set_index("id").loc[r["id"]]
        rows.append(dict(clean=cl[i], corrupted=cor[i], restored=out[i],
                         label=f"{r['severity_level']}\n{pdt.describe_params(r)}",
                         in_text=("∞" if pd.isna(m.in_psnr) else f"{m.in_psnr:.1f}") + f" dB / {m.in_ssim:.3f}",
                         out_text=f"{m.psnr:.1f} dB / {m.ssim:.3f}"))
    return rows

# Representative = the record closest to the median output PSNR of its (type, severity) group.
rep_ids = []
for t in order:
    levels = ["none"] if t == "clean" else ["low", "medium", "high"]
    for lv in levels:
        g = tdf[(tdf.type == t) & (tdf.severity_level == lv)]
        if t == "clean":            # 3 clean examples: 25th / 50th / 75th percentile
            for q in (0.25, 0.5, 0.75):
                rep_ids.append(g.iloc[(g.psnr - g.psnr.quantile(q)).abs().argsort().iloc[0]]["id"])
        else:
            rep_ids.append(g.iloc[(g.psnr - g.psnr.median()).abs().argsort().iloc[0]]["id"])
rep_ids = list(dict.fromkeys(rep_ids))[:12]
figs = {}
for k, chunk in enumerate([rep_ids[:6], rep_ids[6:12]]):
    p = DIRS["figures"] / f"test_representative_{k + 1}.png"
    fig = R.restoration_grid(rows_for(chunk), p, f"Representative test examples ({k + 1}/2) — labels: PSNR / SSIM")
    plt.show(); figs[f"figures/representative_{k + 1}"] = p

# Failure cases: 2 lowest output SSIM + 2 largest PSNR drops (restoration made it worse), distinct types where possible.
# (Clean images have no ΔPSNR; their degradation is captured by the SSIM criterion and the clean row of the table.)
fail_ids = []
for crit, asc in [("ssim", True), ("d_psnr", True)]:
    cand = tdf.sort_values(crit, ascending=asc)
    used_types = set()
    for _, r in cand.iterrows():
        if r.id not in fail_ids and r.type not in used_types:
            fail_ids.append(r.id); used_types.add(r.type)
        if len(used_types) == 2:
            break
fail_ids = fail_ids[:4]
p = DIRS["figures"] / "test_failure_cases.png"
fig = R.restoration_grid(rows_for(fail_ids), p, "Failure cases — labels: PSNR / SSIM"); plt.show()
figs["figures/failure_cases"] = p
failure_table = tdf.set_index("id").loc[fail_ids, ["type", "severity_level", "in_psnr", "psnr", "d_psnr", "in_ssim", "ssim"]]
failure_table.to_csv(DIRS["results"] / "failure_cases.csv"); display(failure_table.round(3))

In [ ]:
if run:
    run.summary.update({f"test/{k}": v for k, v in ev_test.items() if k != "per_image"})
    run.log({"test/by_type": wandb.Table(dataframe=by_type.reset_index().rename(columns={"index": "condition"})),
             "test/by_type_severity": wandb.Table(dataframe=by_sev.reset_index()),
             "test/failure_cases": wandb.Table(dataframe=failure_table.reset_index())})
    wb_log_figs(run, {**figs, "figures/test_severity_curves": DIRS["figures"] / "test_severity_curves.png"})

## 7. ONNX export, PyTorch ↔ ONNX parity, latency

In [ ]:
ONNX_PATH = DIRS["models"] / "task1_universal_dae.onnx"
model_cpu = build_model(FINAL_HP).cpu(); model_cpu.load_state_dict({k: v.cpu() for k, v in ck["model"].items()}); model_cpu.eval()
R.export_onnx(model_cpu, ONNX_PATH)

vx = torch.cat([b[0] for _, b in zip(range(2), val_loader)])[:64]          # mixed corruption types
tx = torch.cat([b[0] for _, b in zip(range(1), test_loader)])[:64]
parity = {"val": R.verify_onnx(model_cpu, ONNX_PATH, vx), "test": R.verify_onnx(model_cpu, ONNX_PATH, tx)}
latency = R.onnx_latency_ms(ONNX_PATH, n_runs=100 if MODE == "full" else 20)
size_mb = ONNX_PATH.stat().st_size / 1e6
print(json.dumps({"parity": parity, "latency_cpu_batch1": latency, "size_mb": round(size_mb, 2)}, indent=1))
assert parity["val"]["passed"] and parity["test"]["passed"], "ONNX output differs from PyTorch!"

sidecar = {"task": "universal_restoration", "onnx_file": ONNX_PATH.name, "opset": 17,
           "input": {"name": "input", "shape": ["batch", 3, 128, 128], "dtype": "float32", "range": [0, 1],
                     "layout": "NCHW, RGB", "preprocess": "PIL convert('RGB') -> resize 128x128 BICUBIC -> /255"},
           "output": {"name": "output", "shape": ["batch", 3, 128, 128], "range": [0, 1]},
           "model_config": model_cpu.config, "hyperparameters": FINAL_HP, **model_info,
           "best_epoch": int(ck["epoch"]), "val_metrics": ck["val"], "onnx_parity": parity, "cpu_latency_ms": latency}
pdt.save_json(sidecar, DIRS["models"] / "task1_universal_dae.json", indent=1)
if run:
    run.summary.update({"onnx/max_abs_diff": max(parity["val"]["max_abs_diff"], parity["test"]["max_abs_diff"]),
                        "onnx/latency_ms": latency["median_ms"], "onnx/size_mb": size_mb})
    art = wandb.Artifact("task1-universal-dae-onnx", type="model", metadata=sidecar)
    art.add_file(str(ONNX_PATH)); art.add_file(str(DIRS["models"] / "task1_universal_dae.json")); run.log_artifact(art)
    run.finish()
del test_loader; gc.collect()

## 8. Skip-connection ablation: purpose and effect
Four variants are trained with the final hyper-parameters for the same number of epochs:
the pure autoencoder, one bottlenecked skip, two bottlenecked skips, and the **v1 wide-skip design** (full-width features, no 1×1 squeeze).
For each variant, **code-dependence tests** zero the latent or the skip codes at inference and measure the PSNR drop.
* Large `latent_drop_db`: the latent is essential, so it is a genuine autoencoder.
* Near-zero `latent_drop_db`: information bypasses the bottleneck. This is the v1 failure mode.

In [ ]:
sc_final = FINAL_HP.get("skip_channels", 4)
VARIANTS = [("pure AE", 0, sc_final), ("1 bottlenecked skip", 1, sc_final), ("2 bottlenecked skips", 2, sc_final),
            ("2 wide skips (v1 design)", 2, None)]
ablation = []
if RUN_ABLATION:
    for name, s, sc in VARIANTS:
        hp = {**FINAL_HP, "skip_levels": s, "skip_channels": sc}
        torch.manual_seed(SEED)
        m = build_model(hp)
        ld = pdt.make_train_loader(train_images, batch_size=hp["batch_size"], num_workers=NUM_WORKERS)
        o, sch_a = R.make_optimizer_and_scheduler(m, hp["lr"], WEIGHT_DECAY, len(ld), S["ablation_epochs"])
        scal, lf = R.make_scaler(device, AMP), R.L1SSIMLoss(hp["alpha"])
        tag = f"ablation-skip{s}-" + ("wide" if sc is None else f"c{sc}")
        arun = wb_init(tag, "ablation", {**hp, "variant": name, "epochs": S["ablation_epochs"], "code_dim": m.code_dim})
        for ep in range(S["ablation_epochs"]):
            tr = R.train_one_epoch(m, ld, lf, o, sch_a, device, scal, AMP)
            if arun:
                arun.log({"epoch": ep, "train/loss": tr["loss"]})
        ev = R.evaluate(m, val_loader, device, lf, AMP)
        cdp = R.code_dependence(m, val_loader, device)
        row = {"variant": name, "skip_levels": s, "skip_channels": "-" if s == 0 else ("full" if sc is None else str(sc)), "code_dim": m.code_dim,
               "compression": round(m.compression_ratio, 2), "params_M": R.count_params(m) / 1e6,
               "val_score": ev["score"], "val_psnr": ev["psnr"], "val_ssim": ev["ssim"],
               **{f"psnr_{n}": ev[f"psnr/{n}"] for n in pdt.CLASS_NAMES}, **cdp}
        ablation.append(row)
        print(f"{name:26s} code {m.code_dim:>7,} ({m.compression_ratio:5.2f}x) | val PSNR {ev['psnr']:.2f} | "
              f"latent drop {cdp['latent_drop_db']:.2f} dB" + (f" | skip drop {cdp['skip_drop_db']:.2f} dB" if "skip_drop_db" in cdp else ""))
        if arun:
            arun.summary.update({k: v for k, v in row.items() if not isinstance(v, str)}); arun.finish()
        del m, ld, o; gc.collect(); torch.cuda.empty_cache()
    abl_df = pd.DataFrame(ablation).set_index("variant")
    abl_df.to_csv(DIRS["results"] / "skip_ablation.csv"); display(abl_df.round(3))
    arun = wb_init("ablation-summary", "ablation")
    if arun:
        arun.log({"skip_ablation": wandb.Table(dataframe=abl_df.reset_index())}); arun.finish()

## 9. Summary

In [ ]:
summary = {"mode": MODE, "skip_ablation": ablation, "code_dependence_final": dep, "data_source": data.source, "best_trial": best_trial.number, "optuna_best_value": best_trial.value,
           "final_hp": FINAL_HP, **model_info, "best_epoch": best_epoch, "best_val_score": best_score,
           "test_overall": by_type.loc["overall"].round(4).to_dict(), "test_all_corrupted": by_type.loc["all corrupted"].round(4).to_dict(),
           "onnx_max_abs_diff": max(parity["val"]["max_abs_diff"], parity["test"]["max_abs_diff"]),
           "onnx_latency_ms": latency["median_ms"], "unit_tests_passed": int(tests_df.passed.sum())}
pdt.save_json(summary, DIRS["results"] / "summary.json", indent=1)
print(json.dumps(summary, indent=1, default=str))
print("\nOutputs:")
for d in DIRS.values():
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(WORK_DIR)}  ({p.stat().st_size / 1e6:.2f} MB)")
print("\nSMOKE TEST PASSED — switch MODE to 'full' and Save & Run All." if MODE == "smoke" else "\nTASK 1 COMPLETE.")